In [1]:
import json
from sqlalchemy.orm import Session, sessionmaker
from lib.db.models import *
from typing import Any
import re
import unicodedata
from difflib import SequenceMatcher

from lib.db.database import engine
from sqlalchemy import text, select, or_

In [ ]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Conexão OK:", result.scalar())
    
SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)
session = SessionLocal()

In [3]:
artigos = []
with open('data/curriculos/2747150211073176/article_crossref.jsonl', 'r') as f:
    for line in f:
        artigos.append(json.loads(line))

len(artigos)
    

258

In [4]:
autores = []
for artigo in artigos:
    autores.extend(artigo['contributors'])

len(autores)

1402

In [14]:
with open('data/autores/autores.json', 'w') as f:
    json.dump(list(autores), f, indent=4, ensure_ascii=False)

In [3]:
with open('data/autores/autores.json', 'r') as f:
    data = json.load(f)

In [ ]:
author_db = get_or_create_author_canonical(session, contributor_data["author"])

# Helpers

In [4]:
DASH_TRANSLATION = str.maketrans({
    "‐": "-",
    "-": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "―": "-",
})

PT_NAME_PARTICLES = {
    "da", "de", "do", "das", "dos", "del", "della", "di", "du",
    "e", "la", "le", "van", "von", "der", "den"
}


def _none_if_blank(value: Any) -> Optional[str]:
    if value is None:
        return None
    value = str(value).strip()
    return value or None

def validate_orcid(orcid: Optional[str]) -> Optional[str]:
    """
    Normaliza ORCID para o formato 0000-0000-0000-0000.
    Retorna None se inválido.
    """
    orcid = _none_if_blank(orcid)
    if not orcid:
        return None

    orcid = orcid.strip()
    orcid = re.sub(r"^https?://orcid\.org/", "", orcid, flags=re.IGNORECASE)
    orcid = orcid.upper().replace(" ", "")

    digits = orcid.replace("-", "")
    if not re.fullmatch(r"\d{15}[\dX]", digits):
        return None

    total = 0
    for ch in digits[:-1]:
        total = (total + int(ch)) * 2

    remainder = (12 - (total % 11)) % 11
    check_digit = "X" if remainder == 10 else str(remainder)

    if check_digit != digits[-1]:
        return None

    return f"{digits[0:4]}-{digits[4:8]}-{digits[8:12]}-{digits[12:16]}"

def _normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def _normalize_dashes(text: str) -> str:
    return text.translate(DASH_TRANSLATION)

def _strip_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))

def _is_initial_token(token: str) -> bool:
    """
    Exemplos considerados iniciais:
    A.
    A
    M.F.
    C.M.
    H.-O.  (depois de normalizar pontuação, tratamos de forma tolerante)
    """
    token = token.strip()
    if not token:
        return False

    token_no_dash = token.replace("-", "")
    token_no_spaces = token_no_dash.replace(" ", "")

    # A., A, M.F., CM, C.M.
    if re.fullmatch(r"(?:[A-Za-zÀ-ÿ]\.?){1,4}", token_no_spaces):
        return True

    return False

def _smart_case_token(token: str, is_middle: bool = True) -> str:
    if not token:
        return token

    token = _normalize_dashes(token)

    low = _strip_accents(token).lower()
    if is_middle and low in PT_NAME_PARTICLES:
        return low

    if _is_initial_token(token):
        letters = re.findall(r"[A-Za-zÀ-ÿ]", token)
        return ".".join(letter.upper() for letter in letters) + ("." if letters else "")

    if "-" in token:
        parts = token.split("-")
        return "-".join(_smart_case_token(part, is_middle=False) for part in parts)

    return token.title()

def normalize_person_name(name: str) -> str:
    """
    Chave técnica para matching.
    Remove acentos, baixa caixa, normaliza hífens e remove pontuação irrelevante.
    """
    name = _normalize_spaces(_normalize_dashes(name))
    name = _strip_accents(name).lower()

    # mantém letras, números, espaços e hífen
    name = re.sub(r"[^\w\s\-]", " ", name, flags=re.UNICODE)
    name = name.replace("-", " ")
    name = _normalize_spaces(name)

    return name

def canonicalize_display_name(name: str) -> str:
    """
    Padroniza caixa, hífens e espaços para exibição.
    Não remove acentos.
    """
    name = _normalize_spaces(_normalize_dashes(name))
    if not name:
        return name

    tokens = name.split(" ")
    out = []
    for i, token in enumerate(tokens):
        is_middle = 0 < i < len(tokens) - 1
        out.append(_smart_case_token(token, is_middle=is_middle))
    return " ".join(out)

def _payload_name_parts_are_consistent(
    full_name: str,
    given_name: Optional[str],
    family_name: Optional[str],
) -> bool:
    if not given_name or not family_name:
        return False

    full_norm = normalize_person_name(full_name)
    combined_norm = normalize_person_name(f"{given_name} {family_name}")

    return full_norm == combined_norm

def split_full_name(full_name: str) -> tuple[Optional[str], Optional[str]]:
    """
    Heurística simples:
    - family_name = último sobrenome + partículas imediatamente anteriores
    - given_name = resto

    Exemplos:
    Adalberto Luis Val -> (Adalberto Luis, Val)
    Maria de Nazaré Paula da Silva -> (Maria de Nazaré Paula, da Silva)
    Susana Braz-Mota -> (Susana, Braz-Mota)
    """
    full_name = canonicalize_display_name(full_name)
    tokens = full_name.split()

    if not tokens:
        return None, None

    if len(tokens) == 1:
        return tokens[0], None

    family_tokens = [tokens[-1]]
    i = len(tokens) - 2

    while i >= 0 and _strip_accents(tokens[i]).lower() in PT_NAME_PARTICLES:
        family_tokens.insert(0, tokens[i])
        i -= 1

    given_tokens = tokens[:i + 1]

    given_name = " ".join(given_tokens).strip() or None
    family_name = " ".join(family_tokens).strip() or None
    return given_name, family_name

In [5]:
def clean_author_payload(author: dict[str, Any]) -> dict[str, Any]:
    """
    Limpa payload e tenta corrigir given/family quando vierem ruins.
    """
    payload: dict[str, Any] = {
        "full_name": _none_if_blank(author.get("full_name")),
        "given_name": _none_if_blank(author.get("given_name")),
        "family_name": _none_if_blank(author.get("family_name")),
        "orcid": validate_orcid(author.get("orcid")),
        "lattes_id": _none_if_blank(author.get("lattes_id")),
        "is_inpa_researcher": author.get("is_inpa_researcher"),
        "affiliation_id": author.get("affiliation_id"),
    }

    if not payload["full_name"]:
        raise ValueError("author.full_name é obrigatório.")

    payload["full_name"] = canonicalize_display_name(payload["full_name"])

    if payload["given_name"]:
        payload["given_name"] = canonicalize_display_name(payload["given_name"])
    if payload["family_name"]:
        payload["family_name"] = canonicalize_display_name(payload["family_name"])

    if not _payload_name_parts_are_consistent(
        payload["full_name"],
        payload["given_name"],
        payload["family_name"],
    ):
        inferred_given, inferred_family = split_full_name(payload["full_name"])
        payload["given_name"] = inferred_given
        payload["family_name"] = inferred_family

    return payload

In [6]:
def _last_non_particle_token(tokens: list[str]) -> Optional[str]:
    for token in reversed(tokens):
        if token not in PT_NAME_PARTICLES:
            return token
    return tokens[-1] if tokens else None

def _first_non_particle_token(tokens: list[str]) -> Optional[str]:
    for token in tokens:
        if token not in PT_NAME_PARTICLES:
            return token
    return tokens[0] if tokens else None

def _build_initials_key_from_parts(given_name: Optional[str], family_name: Optional[str]) -> Optional[str]:
    if not given_name and not family_name:
        return None

    given_tokens = [
        tok for tok in normalize_person_name(given_name or "").split()
        if tok and tok not in PT_NAME_PARTICLES
    ]
    family_tokens = [
        tok for tok in normalize_person_name(family_name or "").split()
        if tok
    ]

    initials = [tok[0] for tok in given_tokens if tok]
    if not family_tokens:
        return " ".join(initials) if initials else None

    return _normalize_spaces(" ".join(initials + family_tokens))

def _name_contains_initials(name: str) -> bool:
    return any(_is_initial_token(tok) for tok in canonicalize_display_name(name).split())

def build_name_keys(author: dict[str, Any]) -> dict[str, Any]:
    """
    Gera chaves auxiliares para matching.
    """
    full_name = author["full_name"]
    given_name = author.get("given_name")
    family_name = author.get("family_name")

    full_norm = normalize_person_name(full_name)
    given_norm = normalize_person_name(given_name or "")
    family_norm = normalize_person_name(family_name or "")

    full_tokens = [tok for tok in full_norm.split() if tok]
    given_tokens = [tok for tok in given_norm.split() if tok]
    family_tokens = [tok for tok in family_norm.split() if tok]

    surname_key = _last_non_particle_token(family_tokens or full_tokens)
    first_given_key = _first_non_particle_token(given_tokens or full_tokens[:-1] or full_tokens)
    first_given_initial = first_given_key[0] if first_given_key else None

    initials_key = _build_initials_key_from_parts(given_name, family_name)
    has_initials = _name_contains_initials(full_name)

    return {
        "full_norm": full_norm,
        "given_norm": given_norm,
        "family_norm": family_norm,
        "tokens": full_tokens,
        "given_tokens": given_tokens,
        "family_tokens": family_tokens,
        "surname_key": surname_key,
        "first_given_key": first_given_key,
        "first_given_initial": first_given_initial,
        "initials_key": initials_key,
        "has_initials": has_initials,
    }

In [7]:
def find_author_by_strong_ids(session: Session, author: dict[str, Any]) -> Optional["Author"]:
    lattes_id = author.get("lattes_id")
    orcid = author.get("orcid")

    by_lattes = None
    by_orcid = None

    if lattes_id:
        by_lattes = session.execute(
            select(Author).where(Author.lattes_id == lattes_id)
        ).scalar_one_or_none()

    if orcid:
        by_orcid = session.execute(
            select(Author).where(Author.orcid == orcid)
        ).scalar_one_or_none()

    if by_lattes and by_orcid and by_lattes.id != by_orcid.id:
        raise ValueError(
            f"Conflito de identidade: lattes_id={lattes_id} aponta para Author.id={by_lattes.id}, "
            f"mas orcid={orcid} aponta para Author.id={by_orcid.id}."
        )

    return by_lattes or by_orcid

In [8]:
def _safe_update_unique_orcid(session: Session, existing: "Author", incoming_orcid: Optional[str]) -> None:
    incoming_orcid = validate_orcid(incoming_orcid)
    if not incoming_orcid:
        return

    if existing.orcid == incoming_orcid:
        return

    if existing.orcid and existing.orcid != incoming_orcid:
        raise ValueError(
            f"Conflito de ORCID para Author.id={existing.id}: "
            f"existente={existing.orcid}, novo={incoming_orcid}."
        )

    other = session.execute(
        select(Author).where(Author.orcid == incoming_orcid, Author.id != existing.id)
    ).scalar_one_or_none()

    if other:
        raise ValueError(
            f"ORCID {incoming_orcid} já pertence ao Author.id={other.id}. "
            f"Não foi possível mesclar automaticamente."
        )

    existing.orcid = incoming_orcid
    
def _safe_update_unique_lattes(session: Session, existing: "Author", incoming_lattes: Optional[str]) -> None:
    incoming_lattes = _none_if_blank(incoming_lattes)
    if not incoming_lattes:
        return

    if existing.lattes_id == incoming_lattes:
        return

    if existing.lattes_id and existing.lattes_id != incoming_lattes:
        raise ValueError(
            f"Conflito de Lattes ID para Author.id={existing.id}: "
            f"existente={existing.lattes_id}, novo={incoming_lattes}."
        )

    other = session.execute(
        select(Author).where(Author.lattes_id == incoming_lattes, Author.id != existing.id)
    ).scalar_one_or_none()

    if other:
        raise ValueError(
            f"Lattes ID {incoming_lattes} já pertence ao Author.id={other.id}. "
            f"Não foi possível mesclar automaticamente."
        )

    existing.lattes_id = incoming_lattes
    
def _name_quality_score(name: Optional[str]) -> float:
    if not name:
        return -999.0

    raw = _normalize_spaces(_normalize_dashes(name))
    canonical = canonicalize_display_name(raw)
    tokens = canonical.split()

    expanded_tokens = sum(1 for tok in tokens if not _is_initial_token(tok))
    initial_tokens = sum(1 for tok in tokens if _is_initial_token(tok))
    has_accents = raw != _strip_accents(raw)
    all_upper = raw.isupper()

    score = 0.0
    score += expanded_tokens * 10
    score -= initial_tokens * 2
    score += len(raw) * 0.05
    score += 1.0 if has_accents else 0.0
    score -= 2.0 if all_upper else 0.0

    return score

def choose_better_name_variant(current_name: Optional[str], incoming_name: Optional[str]) -> Optional[str]:
    current_name = _none_if_blank(current_name)
    incoming_name = _none_if_blank(incoming_name)

    if not current_name:
        return canonicalize_display_name(incoming_name) if incoming_name else None
    if not incoming_name:
        return canonicalize_display_name(current_name)

    current_clean = canonicalize_display_name(current_name)
    incoming_clean = canonicalize_display_name(incoming_name)

    current_score = _name_quality_score(current_clean)
    incoming_score = _name_quality_score(incoming_clean)

    return incoming_clean if incoming_score > current_score else current_clean
    
def merge_author_data(session: Session, existing: "Author", incoming: dict[str, Any]) -> "Author":
    """
    Enriquece o autor existente com dados melhores.
    """
    incoming = clean_author_payload(incoming)

    _safe_update_unique_orcid(session, existing, incoming.get("orcid"))
    _safe_update_unique_lattes(session, existing, incoming.get("lattes_id"))

    existing.full_name = choose_better_name_variant(existing.full_name, incoming.get("full_name"))

    if not existing.given_name and incoming.get("given_name"):
        existing.given_name = incoming["given_name"]
    elif existing.given_name and incoming.get("given_name"):
        current_score = _name_quality_score(existing.given_name)
        incoming_score = _name_quality_score(incoming["given_name"])
        if incoming_score > current_score:
            existing.given_name = incoming["given_name"]

    if not existing.family_name and incoming.get("family_name"):
        existing.family_name = incoming["family_name"]
    elif existing.family_name and incoming.get("family_name"):
        current_score = _name_quality_score(existing.family_name)
        incoming_score = _name_quality_score(incoming["family_name"])
        if incoming_score > current_score:
            existing.family_name = incoming["family_name"]

    if existing.is_inpa_researcher is None and incoming.get("is_inpa_researcher") is not None:
        existing.is_inpa_researcher = incoming["is_inpa_researcher"]
    elif incoming.get("is_inpa_researcher") is True:
        existing.is_inpa_researcher = True

    if getattr(existing, "affiliation_id", None) is None and incoming.get("affiliation_id") is not None:
        existing.affiliation_id = incoming["affiliation_id"]

    session.flush()
    return existing

In [9]:
def _dedupe_authors(authors: list["Author"]) -> list["Author"]:
    seen_ids = set()
    out = []
    for author in authors:
        if author.id not in seen_ids:
            out.append(author)
            seen_ids.add(author.id)
    return out

def find_author_candidates(session: Session, author: dict[str, Any], keys: dict[str, Any]) -> list["Author"]:
    """
    Busca candidatos por nome.
    Compatível com o model atual, sem coluna normalized_full_name.
    """
    clauses = [Author.full_name == author["full_name"]]

    family_name = author.get("family_name")
    if family_name:
        clauses.append(Author.family_name == family_name)

    surname_key = keys.get("surname_key")
    if surname_key and len(surname_key) >= 3:
        clauses.append(Author.full_name.ilike(f"%{surname_key}%"))
        clauses.append(Author.family_name.ilike(f"%{surname_key}%"))

    first_given_key = keys.get("first_given_key")
    if first_given_key and len(first_given_key) >= 2:
        clauses.append(Author.full_name.ilike(f"%{first_given_key}%"))
        clauses.append(Author.given_name.ilike(f"%{first_given_key}%"))

    stmt = select(Author).where(or_(*clauses)).limit(100)
    candidates = session.execute(stmt).scalars().all()

    return _dedupe_authors(candidates)

In [10]:
def _author_to_payload(author_db: "Author") -> dict[str, Any]:
    return {
        "full_name": author_db.full_name,
        "given_name": author_db.given_name,
        "family_name": author_db.family_name,
        "orcid": author_db.orcid,
        "lattes_id": author_db.lattes_id,
        "is_inpa_researcher": author_db.is_inpa_researcher,
        "affiliation_id": getattr(author_db, "affiliation_id", None),
    }
    
def _token_overlap_ratio(tokens_a: list[str], tokens_b: list[str]) -> float:
    if not tokens_a or not tokens_b:
        return 0.0

    set_a = {t for t in tokens_a if t not in PT_NAME_PARTICLES}
    set_b = {t for t in tokens_b if t not in PT_NAME_PARTICLES}

    if not set_a or not set_b:
        return 0.0

    return len(set_a & set_b) / max(len(set_a), len(set_b))
    
def score_author_candidate(
    incoming_author: dict[str, Any],
    candidate: "Author",
    incoming_keys: dict[str, Any],
) -> float:
    """
    Score de 0.0 a 1.0.

    Regras:
    - mesmo lattes_id ou orcid => 1.0
    - mesmo nome normalizado => muito alto
    - mesmo sobrenome + mesmas iniciais => alto
    - apenas mesmo sobrenome => insuficiente
    """
    if incoming_author.get("lattes_id") and candidate.lattes_id == incoming_author["lattes_id"]:
        return 1.0

    if incoming_author.get("orcid") and candidate.orcid == incoming_author["orcid"]:
        return 1.0

    candidate_payload = _author_to_payload(candidate)
    candidate_keys = build_name_keys(candidate_payload)

    if incoming_keys["full_norm"] == candidate_keys["full_norm"]:
        return 0.97

    score = 0.0

    same_surname = (
        incoming_keys.get("surname_key")
        and candidate_keys.get("surname_key")
        and incoming_keys["surname_key"] == candidate_keys["surname_key"]
    )
    if same_surname:
        score += 0.35

    same_first_given = (
        incoming_keys.get("first_given_key")
        and candidate_keys.get("first_given_key")
        and incoming_keys["first_given_key"] == candidate_keys["first_given_key"]
    )
    if same_first_given:
        score += 0.20

    same_first_initial = (
        incoming_keys.get("first_given_initial")
        and candidate_keys.get("first_given_initial")
        and incoming_keys["first_given_initial"] == candidate_keys["first_given_initial"]
    )
    if same_first_initial and (incoming_keys["has_initials"] or candidate_keys["has_initials"]):
        score += 0.10

    same_initials_key = (
        incoming_keys.get("initials_key")
        and candidate_keys.get("initials_key")
        and incoming_keys["initials_key"] == candidate_keys["initials_key"]
    )
    if same_initials_key:
        if incoming_keys["has_initials"] or candidate_keys["has_initials"]:
            score += 0.35
        elif same_first_given:
            score += 0.20
        else:
            score += 0.05

    overlap = _token_overlap_ratio(incoming_keys["tokens"], candidate_keys["tokens"])
    score += overlap * 0.20

    ratio = SequenceMatcher(
        None,
        incoming_keys["full_norm"],
        candidate_keys["full_norm"],
    ).ratio()
    score += ratio * 0.10

    # Sem mesmo sobrenome, não deixa passar de score de merge automático
    if not same_surname:
        score = min(score, 0.84)

    return round(min(score, 0.99), 3)

In [11]:
def create_author(session: Session, author: dict[str, Any]) -> "Author":
    author = clean_author_payload(author)

    author_db = Author(**author)
    session.add(author_db)
    session.flush()
    return author_db

# Main

In [12]:
def get_or_create_author_canonical(session: Session, author: dict[str, Any]) -> "Author":
    """
    Pipeline principal de canonização.

    Decisão:
    - lattes_id/orcid -> merge direto
    - nome igual ou score alto -> merge
    - score intermediário -> merge conservador só quando nome é bem compatível
    - sem candidato bom -> cria novo
    """
    incoming = clean_author_payload(author)
    incoming_keys = build_name_keys(incoming)
    
    # 1. Identificadores fortes
    strong_match = find_author_by_strong_ids(session, incoming)
    if strong_match:
        return merge_author_data(session, strong_match, incoming)
    
    # 2. Candidatos por nome
    candidates = find_author_candidates(session, incoming, incoming_keys)
    if not candidates:
        print(f"Nenhum candidato encontrado para '{incoming['full_name']}'")
        return create_author(session, incoming)
    best_candidate = None
    best_score = 0.0

    for candidate in candidates:
        score = score_author_candidate(incoming, candidate, incoming_keys)
        if score > best_score:
            best_score = score
            best_candidate = candidate
            
    # 3. Regras de decisão
    if best_candidate is not None:
        # merge automático forte
        if best_score >= 0.95:
            return merge_author_data(session, best_candidate, incoming)
        
        # merge conservador
        if best_score >= 0.90:
            return merge_author_data(session, best_candidate, incoming)

        # score intermediário: só aceita quando nome completo normalizado é muito parecido
        candidate_payload = _author_to_payload(best_candidate)
        candidate_keys = build_name_keys(candidate_payload)

        similarity = SequenceMatcher(
            None,
            incoming_keys["full_norm"],
            candidate_keys["full_norm"],
        ).ratio()

        if best_score >= 0.85 and similarity >= 0.90:
            return merge_author_data(session, best_candidate, incoming)
  
    # 4. Sem candidato confiável -> cria novo
    return create_author(session, incoming)

In [17]:
for i in data:
    author = i['author']
    author_db = get_or_create_author_canonical(session, author)
    print(f"Processando '{author['full_name']}'...")

2026-04-12 18:42:39,452 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 18:42:39,498 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.affiliation_id 
FROM authors 
WHERE authors.orcid = %(orcid_1)s
2026-04-12 18:42:39,499 INFO sqlalchemy.engine.Engine [generated in 0.00098s] {'orcid_1': '0000-0001-9509-1678'}
2026-04-12 18:42:39,504 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.affiliation_id 
FROM authors 
WHERE authors.full_name = %(full_name_1)s OR authors.family_name = %(family_name_1)s OR lower(authors.full_name) LIKE lower(%(full_name_2)s) OR lower(authors.family_name) LIKE lower(%(family_name_2)s) OR lower(authors.full_name) LIKE lower(%(full_name_3)s) OR lower(authors.given_name) LIKE lower(%(given_name_1)s) 


In [18]:
stm = select(Author)
autores = session.execute(stm).scalars().all()
len(autores)

2026-04-12 18:46:28,147 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.affiliation_id 
FROM authors
2026-04-12 18:46:28,150 INFO sqlalchemy.engine.Engine [generated in 0.00219s] {}


559

In [ ]:
def get_or_create_author_canonical(session: Session, author: dict[str, Any]) -> "Author":
    """
    Pipeline principal de canonização.

    Decisão:
    - lattes_id/orcid -> merge direto
    - nome igual ou score alto -> merge
    - score intermediário -> merge conservador só quando nome é bem compatível
    - sem candidato bom -> cria novo
    """
    incoming = clean_author_payload(author)
    incoming_keys = build_name_keys(incoming)

    # 1. Identificadores fortes
    strong_match = find_author_by_strong_ids(session, incoming)
    if strong_match:
        return merge_author_data(session, strong_match, incoming)

    # 2. Candidatos por nome
    candidates = find_author_candidates(session, incoming, incoming_keys)

    if not candidates:
        return create_author(session, incoming)

    best_candidate = None
    best_score = 0.0

    for candidate in candidates:
        score = score_author_candidate(incoming, candidate, incoming_keys)
        if score > best_score:
            best_score = score
            best_candidate = candidate

    # 3. Regras de decisão
    if best_candidate is not None:
        # merge automático forte
        if best_score >= 0.95:
            return merge_author_data(session, best_candidate, incoming)

        # merge conservador
        if best_score >= 0.90:
            return merge_author_data(session, best_candidate, incoming)

        # score intermediário: só aceita quando nome completo normalizado é muito parecido
        candidate_payload = _author_to_payload(best_candidate)
        candidate_keys = build_name_keys(candidate_payload)

        similarity = SequenceMatcher(
            None,
            incoming_keys["full_norm"],
            candidate_keys["full_norm"],
        ).ratio()

        if best_score >= 0.85 and similarity >= 0.90:
            return merge_author_data(session, best_candidate, incoming)

    # 4. Sem candidato confiável -> cria novo
    return create_author(session, incoming)

In [ ]:
from __future__ import annotations

import re
import unicodedata

from typing import Any, Optional

from sqlalchemy import or_, select
from sqlalchemy.orm import Session

# importe teu model real
# from app.models import Author
# from app.models import Affiliation

def clean_author_payload(author: dict[str, Any]) -> dict[str, Any]:
    """
    Limpa payload e tenta corrigir given/family quando vierem ruins.
    """
    payload: dict[str, Any] = {
        "full_name": _none_if_blank(author.get("full_name")),
        "given_name": _none_if_blank(author.get("given_name")),
        "family_name": _none_if_blank(author.get("family_name")),
        "orcid": validate_orcid(author.get("orcid")),
        "lattes_id": _none_if_blank(author.get("lattes_id")),
        "is_inpa_researcher": author.get("is_inpa_researcher"),
        "affiliation_id": author.get("affiliation_id"),
    }

    if not payload["full_name"]:
        raise ValueError("author.full_name é obrigatório.")

    payload["full_name"] = canonicalize_display_name(payload["full_name"])

    if payload["given_name"]:
        payload["given_name"] = canonicalize_display_name(payload["given_name"])
    if payload["family_name"]:
        payload["family_name"] = canonicalize_display_name(payload["family_name"])

    if not _payload_name_parts_are_consistent(
        payload["full_name"],
        payload["given_name"],
        payload["family_name"],
    ):
        inferred_given, inferred_family = split_full_name(payload["full_name"])
        payload["given_name"] = inferred_given
        payload["family_name"] = inferred_family

    return payload
























def get_or_create_author_canonical(session: Session, author: dict[str, Any]) -> "Author":
    """
    Pipeline principal de canonização.

    Decisão:
    - lattes_id/orcid -> merge direto
    - nome igual ou score alto -> merge
    - score intermediário -> merge conservador só quando nome é bem compatível
    - sem candidato bom -> cria novo
    """
    incoming = clean_author_payload(author)
    incoming_keys = build_name_keys(incoming)

    # 1. Identificadores fortes
    strong_match = find_author_by_strong_ids(session, incoming)
    if strong_match:
        return merge_author_data(session, strong_match, incoming)

    # 2. Candidatos por nome
    candidates = find_author_candidates(session, incoming, incoming_keys)

    if not candidates:
        return create_author(session, incoming)

    best_candidate = None
    best_score = 0.0

    for candidate in candidates:
        score = score_author_candidate(incoming, candidate, incoming_keys)
        if score > best_score:
            best_score = score
            best_candidate = candidate

    # 3. Regras de decisão
    if best_candidate is not None:
        # merge automático forte
        if best_score >= 0.95:
            return merge_author_data(session, best_candidate, incoming)

        # merge conservador
        if best_score >= 0.90:
            return merge_author_data(session, best_candidate, incoming)

        # score intermediário: só aceita quando nome completo normalizado é muito parecido
        candidate_payload = _author_to_payload(best_candidate)
        candidate_keys = build_name_keys(candidate_payload)

        similarity = SequenceMatcher(
            None,
            incoming_keys["full_norm"],
            candidate_keys["full_norm"],
        ).ratio()

        if best_score >= 0.85 and similarity >= 0.90:
            return merge_author_data(session, best_candidate, incoming)

    # 4. Sem candidato confiável -> cria novo
    return create_author(session, incoming)